# Анализе истренираних модела — GradCAM, аблација, робусност

Ова свеска **ништа не тренира**. Само учитава већ сачуване тежине са Drive-а и
пропушта тест скуп кроз њих, па јој **GPU није потребан**.

Пре покретања подеси: **Runtime → Change runtime type → Hardware accelerator → CPU**.
Тако се не троши GPU квота.

Покретати ћелије редом, одозго надоле. После сваке анализе резултат одмах одлази
на Drive, па прекид сесије не значи губитак посла.


## 1. Провера окружења

In [ ]:
import os, torch, multiprocessing
UREDJAJ = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('uredjaj:', UREDJAJ)
print('jezgara:', multiprocessing.cpu_count())
if UREDJAJ == 'cpu':
    print('\nRadi se na procesoru - to je u redu, analize ne treniraju nista.')


## 2. Повезивање Drive-а

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Распакивање кода и података

In [ ]:
import shutil, zipfile, pathlib

MOJ_DRIVE = pathlib.Path('/content/drive/MyDrive')
kandidati = [d for d in MOJ_DRIVE.iterdir() if d.is_dir() and d.name.lower() == 'diplomski']
assert kandidati, f"Nema foldera 'diplomski' u {MOJ_DRIVE}"
DRIVE = kandidati[0]
print('Drive folder:', DRIVE)

PROJEKAT = pathlib.Path('/content/galaksije')
shutil.rmtree(PROJEKAT, ignore_errors=True)
ODREDISTA = {'kod.zip': PROJEKAT, 'podaci.zip': PROJEKAT / 'data'}


def raspakuj(zip_putanja: pathlib.Path, odrediste: pathlib.Path) -> int:
    """Prevodi obrnutu kosu crtu iz Windows zipa u '/', pa snima."""
    broj = 0
    with zipfile.ZipFile(zip_putanja) as z:
        for clan in z.infolist():
            ime = clan.filename.replace('\\', '/')
            if ime.endswith('/') or '__pycache__' in ime or not ime.strip():
                continue
            cilj = odrediste / ime
            cilj.parent.mkdir(parents=True, exist_ok=True)
            with z.open(clan) as ulaz, open(cilj, 'wb') as izlaz:
                shutil.copyfileobj(ulaz, izlaz)
            broj += 1
    return broj


for naziv, odrediste in ODREDISTA.items():
    izvor = DRIVE / naziv
    assert izvor.exists(), f'Nedostaje {izvor}. Na Drive-u ima: {sorted(f.name for f in DRIVE.iterdir())}'
    odrediste.mkdir(parents=True, exist_ok=True)
    print(f'{naziv}: {raspakuj(izvor, odrediste)} fajlova')

os.chdir(PROJEKAT)
print('\nradni folder:', os.getcwd())
print(sorted(p.name for p in PROJEKAT.iterdir()))


## 4. Инсталација библиотека

In [ ]:
!pip install -q timm
import timm; print('timm', timm.__version__)


## 5. Враћање тежина са Drive-а

Четрнаест зипова са прошлог тренинга се распакује назад на локални диск.

In [ ]:
import zipfile, pathlib, shutil

IZLAZ_LOKALNO = pathlib.Path('/content/outputs_500_pretrained')
IZLAZ_DRIVE   = DRIVE / 'outputs_500_pretrained'
ANALIZE       = pathlib.Path('/content/analize')
ANALIZE.mkdir(parents=True, exist_ok=True)
IZLAZ_LOKALNO.mkdir(parents=True, exist_ok=True)

zipovi = sorted(IZLAZ_DRIVE.glob('*.zip'))
print(f'zipova na Drive-u: {len(zipovi)}')
for z in zipovi:
    with zipfile.ZipFile(z) as f:
        f.extractall(IZLAZ_LOKALNO)

# summary.csv, ako je sacuvan
if (IZLAZ_DRIVE / 'summary.csv').exists():
    shutil.copy2(IZLAZ_DRIVE / 'summary.csv', IZLAZ_LOKALNO / 'summary.csv')

runovi = sorted(p for p in IZLAZ_LOKALNO.iterdir() if p.is_dir())
print(f'\n{"run":<44} best.ckpt   MB')
print('-' * 68)
nedostaje = []
for r in runovi:
    ck = r / 'best.ckpt'
    if ck.exists():
        print(f'{r.name:<44} OK      {ck.stat().st_size/1e6:7.1f}')
    else:
        print(f'{r.name:<44} NEMA!')
        nedostaje.append(r.name)

print(f'\nukupno runova: {len(runovi)}')
assert len(runovi) == 14 and not nedostaje, f'Fali tezina: {nedostaje}'
print('Sve tezine su na mestu.')

# Sitni fajlovi sa brojevima (metrics.json, summary.csv) idu uz analize,
# da bi se tabele u radu mogle popuniti bez skidanja 500 MB tezina.
METRIKE = ANALIZE / 'metrike'
METRIKE.mkdir(exist_ok=True)
for r in runovi:
    if (r / 'metrics.json').exists():
        shutil.copy2(r / 'metrics.json', METRIKE / f'{r.name}.json')
if (IZLAZ_LOKALNO / 'summary.csv').exists():
    shutil.copy2(IZLAZ_LOKALNO / 'summary.csv', METRIKE / 'summary.csv')
print(f'metrika prikupljeno: {len(list(METRIKE.iterdir()))} fajlova')



def posalji_na_drajv(naziv='analize.zip'):
    """Pakuje trenutni sadrzaj /content/analize i proverava da je stigao."""
    privremeni = pathlib.Path('/content') / naziv
    with zipfile.ZipFile(privremeni, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in sorted(ANALIZE.rglob('*')):
            if f.is_file():
                z.write(f, f.relative_to(ANALIZE))
    ocekivano = privremeni.stat().st_size
    odrediste = DRIVE / naziv
    shutil.copy2(privremeni, odrediste)
    stiglo = odrediste.stat().st_size if odrediste.exists() else 0
    print(f'{naziv} -> Drive: {stiglo/1e6:.2f} MB  {"OK" if stiglo == ocekivano else "NIJE PRENET!"}')
    return stiglo == ocekivano

# Врати и оно што је већ урађено у ранијим сесијама (GradCAM слике), да их
# ново слање на Drive не препише. analize.zip се допуњује, не замењује.
stari = DRIVE / 'analize.zip'
if stari.exists():
    with zipfile.ZipFile(stari) as z:
        z.extractall(ANALIZE)
    print(f'враћено из ранијег analize.zip: {len(list(ANALIZE.rglob("*.png")))} слика')
else:
    print('нема ранијег analize.zip на Drive-у')


## 6. Процена трајања

Мери се једно стварно пропуштање кроз мрежу на овом рачунару, па се одатле
рачуна колико ће свака анализа трајати. На графичком процесору ово је реда
десетак минута по анализи; на процесору вишеструко дуже.


In [ ]:
import time, torch
from src.gz_classifier.data import build_local_label_info
from src.gz_classifier.evaluacija import napravi_loader
from src.gz_classifier.infer import load_predictor

PODACI = 'data/gz_hubble_500'
label_info = build_local_label_info(PODACI)
BROJ_TEST = len(napravi_loader(PODACI, 'test', label_info, ['original'], 224,
                               batch_size=1, num_workers=0).dataset)
print('slika u test skupu:', BROJ_TEST)

vreme_po_slici = {}
for model in ['densenet121', 'swin_tiny_patch4_window7_224']:
    pred = load_predictor(IZLAZ_LOKALNO / f'{model}_multimodal' / 'best.ckpt', device=UREDJAJ)
    loader = napravi_loader(PODACI, 'test', label_info, list(pred.modalities),
                            pred.image_size, batch_size=16, num_workers=2, max_slika=32)
    with torch.inference_mode():
        for x, _ in loader:                      # prvi prolaz se ne meri (zagrevanje)
            pred.model(x.to(pred.device)); break
        pocetak, n = time.time(), 0
        for x, _ in loader:
            pred.model(x.to(pred.device)); n += len(x)
    vreme_po_slici[model] = (time.time() - pocetak) / n
    print(f'{model:<34} {1000*vreme_po_slici[model]:6.1f} ms po slici')
    del pred

d, s = vreme_po_slici['densenet121'], vreme_po_slici['swin_tiny_patch4_window7_224']
prosek = (d + s) / 2

procene = {
    'GradCAM slike + doprinos':  2 * BROJ_TEST * prosek * 3,      # unapred + unazad
    'Ablacija (7 modela x 7)':   7 * 7 * BROJ_TEST * prosek,
    'Robusnost (2 modela x 2)':  2 * 2 * 14 * BROJ_TEST * prosek,
}
print()
ukupno = 0
for naziv, sek in procene.items():
    ukupno += sek
    print(f'{naziv:<28} ~{sek/60:5.1f} min')
print(f'{"UKUPNO":<28} ~{ukupno/60:5.1f} min')


## 7. GradCAM слике — ВЕЋ УРАЂЕНО

Ове слике су направљене у претходној сесији и налазе се у `analize.zip` на
Drive-у, а ћелија 5 их је вратила назад. **Прескочи ове две ћелије.**

Покрени их поново само ако си променила нешто у `slike_gradcam.py`.


In [ ]:
!python slike_gradcam.py \
    --output-root /content/outputs_500_pretrained \
    --data-dir data/gz_hubble_500 \
    --models densenet121 swin_tiny_patch4_window7_224 \
    --per-class 2 \
    --device $UREDJAJ \
    --out /content/analize


In [ ]:
posalji_na_drajv('analize.zip')
for f in sorted(ANALIZE.iterdir()):
    print(f'  {f.name:<44} {f.stat().st_size/1e3:8.1f} KB')


## 8. Аблација модалитета — ПОКРЕНИ ОВО

Свих седам мултимодалних модела. За сваки се тест скуп најпре пропушта са свим
модалитетима, па још шест пута, сваки пут са по једним искљученим. Пад макро F1
мере показује колико је тај модалитет заиста доприносио тачности.

То је одговор на питање које тежине адаптера не могу да дају: оне мере колико
је простора модел дао модалитету, а ово мери колико му је стварно користио.

Траје око десет минута на графичком процесору.


In [ ]:
!python ablacija.py \
    --output-root /content/outputs_500_pretrained \
    --data-dir data/gz_hubble_500 \
    --device $UREDJAJ \
    --out /content/analize


In [ ]:
posalji_na_drajv('analize.zip')


## 9. Робусност на деградацију слике — ОПЦИОНО, КАСНИЈЕ

Два модела × два услова × четрнаест нивоа кварења слике. Око дванаест минута
на графичком процесору.

Ова анализа ниједан закључак у раду не мења — она додаје ново поглавље.
Покренути је тек ако остане времена после консултација.


In [ ]:
!python robusnost.py \
    --output-root /content/outputs_500_pretrained \
    --data-dir data/gz_hubble_500 \
    --models densenet121 swin_tiny_patch4_window7_224 \
    --device $UREDJAJ \
    --out /content/analize


In [ ]:
posalji_na_drajv('analize.zip')


## 10. Завршна провера

In [ ]:
# Шта све има у /content/analize, груписано по врсти
grupe = {
    'GradCAM слике':      sorted(f.name for f in ANALIZE.glob('gradcam_*.png')),
    'Допринос по класама': sorted(f.name for f in ANALIZE.glob('doprinos_*')),
    'Аблација':           sorted(f.name for f in ANALIZE.glob('ablacija.*')),
    'Робусност':          sorted(f.name for f in ANALIZE.glob('robusnost*')),
    'Метрике':            sorted(f.name for f in (ANALIZE / 'metrike').glob('*')),
}
for naziv, fajlovi in grupe.items():
    print(f'{naziv:<22} {len(fajlovi):>2} фајлова' + ('' if fajlovi else '   — није рађено'))
    for f in fajlovi[:4]:
        print(f'                       {f}')
    if len(fajlovi) > 4:
        print(f'                       ... и још {len(fajlovi)-4}')

print()
posalji_na_drajv('analize.zip')
print('\nСкини analize.zip са Drive-а и распакуј га у пројекат, преко постојећег фолдера analize.')
